In [ ]:
##Ejercicio 3.1: groupBy + agg —

In [33]:
from pyspark.sql.functions import col,count,avg,min,max

df_fact_props=spark.table("bootcamp.gold.fact_propiedades")



df=(
    df_fact_props
    .groupBy(col("zona_id"))
    .agg(
        count("*").alias("total"),
        avg(col("precio")).alias("avg_precio"),
        min(col("precio")).alias("min_precio"),
        max(col("precio")).alias("max_precio")
        )   
    .orderBy(col("total").desc())
    .limit(5)
)


df.explain(True)

#df.show()


== Parsed Logical Plan ==
'GlobalLimit 5
+- 'LocalLimit 5
   +- 'Sort ['total DESC NULLS LAST], true
      +- 'Aggregate ['zona_id], ['zona_id, 'count(*) AS total#12368, 'avg('precio) AS avg_precio#12369, 'min('precio) AS min_precio#12370, 'max('precio) AS max_precio#12371]
         +- 'UnresolvedRelation [bootcamp, gold, fact_propiedades], [], false

== Analyzed Logical Plan ==
zona_id: bigint, total: bigint, avg_precio: decimal(19,6), min_precio: decimal(15,2), max_precio: decimal(15,2)
GlobalLimit 5
+- LocalLimit 5
   +- Sort [total#12368L DESC NULLS LAST], true
      +- Aggregate [zona_id#12387L], [zona_id#12387L, count(1) AS total#12368L, avg(precio#12393) AS avg_precio#12369, min(precio#12393) AS min_precio#12370, max(precio#12393) AS max_precio#12371]
         +- SubqueryAlias bootcamp.gold.fact_propiedades
            +- Relation bootcamp.gold.fact_propiedades[row_hash#12386,zona_id#12387L,tipo_orientacion_id#12388L,fecha_id#12389L,caracteristicas_id#12390L,tipo_operacion_id#12

In [41]:
from pyspark.sql.functions import col

df_fact_props=spark.table("bootcamp.gold.fact_propiedades")

df_zona=spark.table("bootcamp.gold.dim_zona")

df_result=(

    df_fact_props
    .join(df_zona, on="zona_id", how="inner")
    .select("partido", "region","precio","precio_por_m2")
    .limit(10)
)

df_result.explain(True)

== Parsed Logical Plan ==
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Project ['partido, 'region, 'precio, 'precio_por_m2]
      +- 'Join UsingJoin(Inner, [zona_id])
         :- 'UnresolvedRelation [bootcamp, gold, fact_propiedades], [], false
         +- 'UnresolvedRelation [bootcamp, gold, dim_zona], [], false

== Analyzed Logical Plan ==
partido: string, region: string, precio: decimal(15,2), precio_por_m2: decimal(15,2)
GlobalLimit 10
+- LocalLimit 10
   +- Project [partido#11518, region#11519, precio#11503, precio_por_m2#11505]
      +- Project [zona_id#11497L, row_hash#11496, tipo_orientacion_id#11498L, fecha_id#11499L, caracteristicas_id#11500L, tipo_operacion_id#11501L, url#11502, precio#11503, expensas#11504, precio_por_m2#11505, metros_cuadrados_totales#11506, metros_cuadrados_cubiertos#11507, ambientes#11508, _createdAt#11509, partido#11518, region#11519, ciudad#11520, provincia#11521, pais#11522, _createdAt#11523]
         +- Join Inner, (zona_id#11497L = zona_id#11517L)
     